Damped Newton method for chi=30.
Start Newton method after 4 RG steps.
Damping with newton_step=0.5 is activated a couple of times (e.g. for i=3)
Here I ran up to i=9, reaching fp_error =3.7034621469049967e-5

In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5 #6e-6
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.03361407516010624 and became 0.0. Index CartesianIndex(1, 16, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.012345916839578114 and became -1.8524388175538877e-9. Index CartesianIndex(15, 2, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.004606248308129038 and became 0.0. Index CartesianIndex(17, 3, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0022689612252432567 and became 0.0. Index CartesianIndex(15, 2, 19, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It wa

In [3]:
for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

1 [1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
7 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
10 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
11 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]


In [4]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

In [5]:
A[1] = truncate_blocks(traj[4], trunc_shape)
for i in 1:30
    println("i=",i)
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.0578969047685301
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  10 eigenvalues converged
│ *  norm of residuals = (1.5429882913104113e-47, 3.837823585681178e-33, 1.069489302886352e-32, 7.420824891071248e-23, 5.543045848328512e-23, 9.613793467300595e-19, 9.613793467300595e-19, 4.416915377192728e-14, 2.1931690551365128e-13, 2.1931690551365128e-13)
└ *  number of operations = 51


EIGENVALUES (INITIAL):
1.985023836398595 + 0.0im
-0.9350407213097853 + 0.0im
-0.9250254743326507 + 0.0im
0.557182023311904 + 0.0im
0.5510720267708678 + 0.0im
0.0007488638957161387 + 0.41927022066857533im
0.0007488638957161387 - 0.41927022066857533im
-0.31063011891592424 + 0.0im
-7.185196956606734e-5 + 0.30938481316582456im
-7.185196956606734e-5 - 0.30938481316582456im
||deltaA[i]||= 0.1340675019501551
newton_step= 1.0
fp_error= 0.027510146837136384
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=2
||R(A[i])-A[i]||= 0.027510146837136384
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  13 eigenvalues converged
│ *  norm of residuals = (7.627718691035267e-53, 4.734315152558397e-39, 2.5788795349027243e-38, 6.060757694158223e-31, 9.84797949915968e-29, 1.103934892911266e-25, 1.103934892911266e-25, 3.949197047852344e-21, 6.812829988081916e-15, 4.738006530699232e-16, 4.738006530699232e-16, 4.228469810066503e-15, 4.228469810066503e-15)
└ *  number of operations = 60


EIGENVALUES (INITIAL):
1.9890632300604403 + 0.0im
-0.9724220876533529 + 0.0im
-0.9614503115569584 + 0.0im
0.6462819694388654 + 0.0im
0.5926477100207731 + 0.0im
-0.0077505460924799895 + 0.49446476339301226im
-0.0077505460924799895 - 0.49446476339301226im
-0.3974129595536 + 0.0im
0.33478807923490744 + 0.0im
0.005738271689132268 + 0.32580825910293004im
0.005738271689132268 - 0.32580825910293004im
-0.11272117171888257 + 0.2666236787634007im
-0.11272117171888257 - 0.2666236787634007im
||deltaA[i]||= 0.07758246046627447
newton_step= 1.0
fp_error= 0.019419158758721317
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=3
||R(A[i])-A[i]||= 0.019419158758721317
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


...

┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.9382160388967222e-53, 4.524267826439619e-39, 7.023039494939924e-39, 7.544989656144179e-33, 2.7605483349512536e-29, 1.3438938862081772e-26, 1.3438938862081772e-26, 1.962725695091429e-17, 1.7711241421559033e-17, 1.8517781905027665e-15, 1.8517781905027665e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9983176562044949 + 0.0im
-0.9944953418203211 + 0.0im
-0.9865616339583081 + 0.0im
0.6843526962876183 + 0.0im
0.5957578109454916 + 0.0im
-0.013918307083075301 + 0.5045510377005747im
-0.013918307083075301 - 0.5045510377005747im
0.35111319086480713 + 0.0im
-0.34521377091663863 + 0.0im
-0.14807201888695698 + 0.24405221694255722im
-0.14807201888695698 - 0.24405221694255722im
||deltaA[i]||= 0.00040876612756772615
newton_step= 1.0
fp_error= 0.00013650312260549052
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=8
||R(A[i])-A[i]||= 0.00013650312260549052
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (2.0432301532271792e-53, 3.399947577243759e-39, 9.797463453047596e-39, 3.583170006031172e-32, 1.1927490456698222e-28, 1.8724756582761248e-26, 1.8724756582761248e-26, 1.632261219802733e-17, 3.361508498802332e-16, 1.5133280270182756e-15, 1.5133280270182756e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9983665318654937 + 0.0im
-0.9948179243345624 + 0.0im
-0.9867590321177232 + 0.0im
0.672819159346802 + 0.0im
0.5955643588742489 + 0.0im
-0.012778339643972634 + 0.5026464268286294im
-0.012778339643972634 - 0.5026464268286294im
0.36088592794666585 + 0.0im
-0.3447079011491329 + 0.0im
-0.1482929121948965 + 0.2443570510228875im
-0.1482929121948965 - 0.2443570510228875im
||deltaA[i]||= 0.0003883801328798756
newton_step= 1.0
fp_error= 3.7034621469049967e-5
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=9
||R(A[i])-A[i]||= 3.7034621469049967e-5
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


LoadError: InterruptException: